# Track Multiplicity Analysis

This notebook analyzes particle tracking data to calculate **detector multiplicity** — the number of particles from each source event that reach the detector cell.

Key analyses:
- **Multiplicity distribution**: Histogram of particles detected per source event
- **Statistical moments**: Mean and variance of the multiplicity distribution
- **VTK export**: Convert tracks to VTK format for 3D visualization in ParaView

## Setup and Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import openmc
import scienceplots

# Configure plotting style
plt.style.use(["science", "notebook", "grid", "high-vis"])

# Output directory
FIGURES_DIR = Path("../outputs/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Load Track Data

Load the particle tracks file containing detailed state information for each particle throughout its history.

In [ ]:
# Load tracks file
tracks_file = Path("../data/track_feature/tracks.h5")
tracks = openmc.Tracks(str(tracks_file))

print(f"Loaded {len(tracks)} source particle tracks")

## Calculate Detector Multiplicity

For each source particle, count how many particles (including progeny) reach the detector cell.

In [ ]:
# Detector cell ID
DETECTOR_CELL_ID = 2

# Calculate multiplicity for each source event
multiplicities = []

for track in tracks:
    detected_particles = 0
    
    # Check each particle track in this source event
    for particle_track in track.particle_tracks:
        # Check if this particle ever entered the detector cell
        for state in particle_track.states:
            if state["cell_id"] == DETECTOR_CELL_ID:
                detected_particles += 1
                break  # Count each particle only once
    
    multiplicities.append(detected_particles)

# Convert to numpy array for analysis
multiplicities = np.array(multiplicities)

print(f"\nMultiplicity distribution:")
print(f"  Min: {multiplicities.min()}")
print(f"  Max: {multiplicities.max()}")
print(f"  Mean: {multiplicities.mean():.3f}")
print(f"  Std: {multiplicities.std():.3f}")

### Statistical Analysis

In [ ]:
# Calculate statistical moments
mean_multiplicity = np.mean(multiplicities)
variance_multiplicity = np.var(multiplicities)
std_multiplicity = np.std(multiplicities)

# Fano factor (variance-to-mean ratio)
fano_factor = variance_multiplicity / mean_multiplicity if mean_multiplicity > 0 else 0

print("\nStatistical Moments:")
print(f"  Mean: {mean_multiplicity:.3f}")
print(f"  Variance: {variance_multiplicity:.3f}")
print(f"  Standard deviation: {std_multiplicity:.3f}")
print(f"  Fano factor (σ²/μ): {fano_factor:.3f}")
print(f"\nNote: Fano factor > 1 indicates super-Poissonian statistics (bunching)")

## Visualize Multiplicity Distribution

In [ ]:
# Create integer bins centered on each multiplicity value
max_mult = int(multiplicities.max())
bins = np.arange(-0.5, max_mult + 1.5, 1.0)

# Create histogram
fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
counts, edges, patches = ax.hist(
    multiplicities, bins=bins, edgecolor="black", alpha=0.7, color="#E27D60"
)

# Formatting
ax.set_xlabel("Particles Detected per Source Event")
ax.set_ylabel("Frequency (Number of Source Events)")
ax.set_title("Detector Multiplicity Distribution")
ax.set_xticks(range(max_mult + 1))
ax.grid(True, alpha=0.3)

# Add statistics text box
stats_text = f"Mean: {mean_multiplicity:.3f}\nVariance: {variance_multiplicity:.3f}\nFano: {fano_factor:.3f}"
ax.text(
    0.95,
    0.95,
    stats_text,
    transform=ax.transAxes,
    fontsize=10,
    verticalalignment="top",
    horizontalalignment="right",
    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "track_multiplicity_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

### Zero-Multiplicity Events

In [ ]:
# Analyze events with zero detections
zero_detections = np.sum(multiplicities == 0)
total_events = len(multiplicities)
detection_efficiency = (total_events - zero_detections) / total_events

print(f"\nDetection Statistics:")
print(f"  Source events with zero detections: {zero_detections}/{total_events} ({zero_detections/total_events*100:.1f}%)")
print(f"  Overall detection efficiency: {detection_efficiency*100:.1f}%")

## Export to VTK Format

Export particle tracks to VTK format for 3D visualization in ParaView or other visualization tools.

In [ ]:
# Export tracks to VTK
vtk_output = Path("../data/track_feature/tracks.vtp")
vtk_output.parent.mkdir(parents=True, exist_ok=True)

vtk_data = tracks.write_to_vtk(vtk_output)

print(f"\nExported particle tracks to: {vtk_output}")
print(f"Open this file in ParaView for 3D visualization")

## Summary

This analysis characterized the detector multiplicity distribution:
- **Mean multiplicity**: Indicates average number of particles detected per source event
- **Variance**: Quantifies spread in the distribution
- **Fano factor**: Reveals statistical properties (super-Poissonian if > 1)
- **VTK export**: Enables detailed 3D visualization of particle trajectories